# Graded Lab

This is not fully the original lab. It is a close replication of it since I was not able to get a already created database migrated to my own environment. 

## Module 3 Assignment - Building RAG systems with a Vector Database

---

In this assignment you will work with [Weaviate API](https://weaviate.io/) to build a tiny RAG system. You will:

- Load a [collection](https://weaviate.io/developers/weaviate/manage-data/collections) for BBC News data.
- Use the Weaviate API to retrieve documents from the vector database.
- Create functions to retrieve data based on Semantic Search, BM25 and Hybrid Search (using RRF) using the Weaviate API.
- Use an LLM to generate responses.

**IMPORTANT**: This assignment assumes you know how to handle simple tasks with collections in the Weaviate API. If you are not familiar with it yet, please read the Ungraded Lab on the Weaviate API! Furthermore, the data you will be working here is already *chunked*. You can get more hands-on experience on chunking reading the Ungraded Lab on chunking!

In [43]:
from dotenv import load_dotenv
import joblib
from tqdm import tqdm
from weaviate.util import generate_uuid5
from weaviate.classes.query import Filter, Rerank


load_dotenv()

True

In [44]:
import os
x = os.environ.get("OPENAI_API_KEY")

In [45]:
!docker ps

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


In [46]:
%%writefile docker-compose.yml
services:
  weaviate:
    image: cr.weaviate.io/semitechnologies/weaviate:1.34.5
    command: >
      --host 0.0.0.0
      --port 8080
      --scheme http
    ports:
      - "8080:8080"    # REST API
      - "50051:50051"  # gRPC API
    volumes:
      - weaviate_data:/var/lib/weaviate
    restart: on-failure:0
    environment:
      QUERY_DEFAULTS_LIMIT: 25
      AUTHENTICATION_ANONYMOUS_ACCESS_ENABLED: 'true'
      PERSISTENCE_DATA_PATH: '/var/lib/weaviate'
      CLUSTER_HOSTNAME: 'node1'
      OPENAI_APIKEY: ${OPENAI_API_KEY}

volumes:
  weaviate_data:

Overwriting docker-compose.yml


In [47]:
!docker-compose up -d

[+] Running 0/1
 ⠋ Network graded_lab_default  Creating                                    0.0s 
[+] Running 2/3
 ✔ Network graded_lab_default       Create...                              0.1s 
 ✔ Volume graded_lab_weaviate_data  C...                                   0.0s 
 ⠋ Container graded_lab-weaviate-1  C...                                   0.0s 
[+] Running 2/3
 ✔ Network graded_lab_default       Create...                              0.1s 
 ✔ Volume graded_lab_weaviate_data  C...                                   0.0s 
 ⠙ Container graded_lab-weaviate-1  C...                                   0.1s 
[+] Running 2/3
 ✔ Network graded_lab_default       Create...                              0.1s 
 ✔ Volume graded_lab_weaviate_data  C...                                   0.0s 
 ⠹ Container graded_lab-weaviate-1  S...                                   0.2s 
[+] Running 2/3
 ✔ Network graded_lab_default       Create...                              0.1s 
 ✔ Volume graded_lab_weaviate

#### Check if weaviate is available

In [48]:
!curl -i http://localhost:8080/v1/.well-known/ready

HTTP/1.1 503 Service Unavailable
Date: Sat, 25 Apr 2026 13:36:32 GMT
Content-Length: 0



## Connect to weaviate

https://academy.weaviate.io/courses/wa101t-py/m3/p3

- using context manager
- or without

In [53]:
import weaviate

with weaviate.connect_to_local(skip_init_checks=True) as client:
    print("Ready:", client.is_ready())
    # do your stuff
# automatically closes connection here


Ready: True


In [54]:
import weaviate
import os

headers = {
    "X-Openai-Api-Key": os.getenv("OPENAI_API_KEY")
}  # Replace with your Cohere API key

client = weaviate.connect_to_local(headers=headers)

assert client.is_ready()

<a id='2-2'></a>
### 2.2 Loading the data

Now, let's load the data. The dataset is structured with the following fields:

- **`title`**: The headline of the article.
- **`pubDate`**: The publication date and time of the article.
- **`guid`**: A unique identifier for the article, commonly used for listing.
- **`link`**: A URL link to access the full article online.
- **`description`**: A brief summary or teaser of the article's content.
- **`article_content`**: The complete text of the article, providing detailed information.

In [55]:
bbc_data = joblib.load('data/bbc_data.joblib')
print(len(bbc_data))

9973


In [56]:
bbc_data[0].keys()

dict_keys(['title', 'pubDate', 'guid', 'link', 'description', 'article_content'])

The `pubDate` is pandas timestamp that is utc unaware. This gives some warnings later on so we preprocess that field before ingesting it in the vector db

In [57]:
type(bbc_data[0]["pubDate"])

pandas.Timestamp

Change the `pubDate` to include a timezone

In [58]:
from datetime import timezone

for doc in bbc_data:
    doc['pubDate'] = doc["pubDate"].to_pydatetime().replace(tzinfo=timezone.utc)

In [59]:
bbc_data[0]["pubDate"]

datetime.datetime(2024, 1, 1, 0, 0, 4, tzinfo=datetime.timezone.utc)

#### Create a collection

In [60]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
import os

if not client.collections.exists("bbc_collection"): 
    print("Create collection")
    client.collections.create(
        name="bbc_collection",
        properties=[
            # Property(name="chunk", data_type=DataType.TEXT),
            Property(name="title", data_type=DataType.TEXT),
            Property(name="description", data_type=DataType.TEXT),
            Property(name="article_content", data_type=DataType.TEXT),
            Property(name="link", data_type=DataType.TEXT),
            Property(name="pubDate", data_type=DataType.DATE),
            #Property(name="chunk_index", data_type=DataType.INT),
        ],
        # Define the vectorizer module
        vector_config=Configure.Vectors.text2vec_openai(model="text-embedding-3-small"),
    )    

Create collection


Set the collection

In [61]:
collection = client.collections.get("bbc_collection")

#client.close()

You can also retrieve all the collections saved:

In [62]:
client.collections.list_all().keys()

dict_keys(['Bbc_collection'])

<a id='2-4'></a>
### 2.4 Adding elements into a Collection

Once you create a collection, you get an empty collection. Now you need to add elements to it. When you add an element, two important steps happen in the background:

1. The information is vectorized (as configured in the collection definition)
2. The HNSW index is updated to optimize search (as you saw in the lectures). This occurs in the backend and you don't see it, but this can make the process take a bit of time

Adding elements is completed using a `collection.batch`, which adds additional useful features. For example, it will let you decide how many objects to send in each batch, handle errors during import, and improve performance by reducing the number of individual network calls. In this example, one element is added at a time, with only a single concurrent request at a time.

You can add a uuid (unique identifier id) to each element you add, so this prevents duplicate entries in your database. 

Let's see in practice!

In [63]:
bbc_data[0].keys()

dict_keys(['title', 'pubDate', 'guid', 'link', 'description', 'article_content'])

In [101]:
# Set up a batch process with specified fixed size and concurrency
with collection.batch.fixed_size(batch_size=1, concurrent_requests=1) as batch:
    # Iterate over a subset of the dataset
    for document in tqdm(bbc_data[:3000]): # tqdm is a library to show progress bars
            # Generate a UUID based on the article_content text for unique identification
            uuid = generate_uuid5(document)

            # Add the object to the batch with properties and UUID. 
            # properties expects a dictionary with the keys being the properties.
            batch.add_object(
                properties=document,
                uuid=uuid,
            )

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3000/3000 [07:55<00:00,  6.31it/s]


In [65]:
result = collection.query.fetch_objects(limit=1)
result.objects[0].properties['title']

'Bob Marley fans Harry and Meghan attend film premiere'

In [66]:
sample_movies = collection.query.fetch_objects(
    limit=2, 
    include_vector=False) # set to True to return embedding vector

if sample_movies.objects:
    print(f"Thera are {len(collection)} entries in the database")
else:
    print("The database is empty")


Thera are 1000 entries in the database


In [67]:
# later on we use this list comprehension to parse the result
[x.properties for x in sample_movies.objects[:1]]

[{'link': 'https://www.bbc.co.uk/news/uk-68086080?at_medium=RSS&at_campaign=KARANGA',
  'description': 'The Duke and Duchess of Sussex appear at event in Jamaica remembering the life of the reggae star.',
  'pubDate': datetime.datetime(2024, 1, 24, 21, 22, 18, tzinfo=datetime.timezone.utc),
  'title': 'Bob Marley fans Harry and Meghan attend film premiere',
  'guid': 'https://www.bbc.co.uk/news/uk-68086080',
  'article_content': 'Meghan and Harry with the Jamaican prime minister Andrew Holness at the film premiere The Duke and Duchess of Sussex have long been fans of Bob Marley\'s "music and message", says a source, as the couple attended a premiere in Jamaica of a film about the reggae singer. Prince Harry and Meghan were photographed with the Jamaican Prime Minister Andrew Holness, at the screening of Bob Marley: One Love. Mr Holness has spoken of Jamaica "moving on" to become a republic. Bob Marley\'s relations were also among the guests at the event in Kingston. The red carpet scre

<a id='ex01'></a>

<a id='3-1'></a>
### 3.1 Metadata filtering

<a id='ex01'></a>
### Exercise 1

In this exercise, you will implement a metadata filtering function. This function will take several inputs: a property (such as `article_content`, `title`, `pubDate`, etc.), the values you want to filter by, the collection you want to search in, and the number of items you want to retrieve.

<details>
<summary style="color: green;">Hint 1</summary>
<p>Remember that to perform filtering based only on metadata, the appropriate method to use is <code>collection.query.fetch_objects</code>.</p>
</details>
<details>
<summary style="color: green;">Hint 2</summary>
<p>When using <code>collection.query.fetch_objects</code>, you must provide the <code>metadata_property</code> as the <code>property</code> and the corresponding <code>Filter</code> object.</p>
</details>
<details>
<summary style="color: green;">Hint 3</summary>
<p>The filter object should be used with the method <code>.by_property</code> for the appropriate property, and <code>.contains_any</code> with the relevant values. A typical call would be <code>Filter.by_property(metadata_property).contains_any(values)</code>.</p>
<p>To limit the results, use <code>limit=limit</code> within the <code>.fetch_objects</code> method.</p>
</details>

In [68]:
# GRADED CELL 

def filter_by_metadata(metadata_property: str, 
                       values: list[str], 
                       collection: "weaviate.collections.collection.sync.Collection" , 
                       limit: int = 5) -> list:
    """
    Retrieves objects from a specified collection based on metadata filtering criteria.

    This function queries a collection within the specified client to fetch objects that match 
    certain metadata criteria. It uses a filter to find objects whose specified 'property' contains 
    any of the given 'values'. The number of objects retrieved is limited by the 'limit' parameter.

    Args:
    metadata_property (str): The name of the metadata property to filter on.
    values (List[str]): A list of values to be matched against the specified property.
    collection_name (weaviate.collections.collection.sync.Collection): The collection to query.
    limit (int, optional): The maximum number of objects to retrieve. Defaults to 5.

    Returns:
    List[Object]: A list of objects from the collection that match the filtering criteria.
    """
    ### START CODE HERE ###
    
    # Retrieve using collection.query.fetch_objects
    
    response = collection.query.fetch_objects(
        # filter the collection on a property any of the item in the values (list)
        filters=Filter.by_property(metadata_property).contains_any(values),
        limit=limit
    )

    ### END CODE HERE ###
    
    response_objects = [x.properties for x in response.objects]
    
    return response_objects

In [69]:
# Let's get an example
res = filter_by_metadata('title', ['Taylor Swift'], collection, limit = 2)
for x in res:
    print(x['title'])

Margot Robbie, Taylor Swift and more on Golden Globes red carpet
TikTok mutes users' videos as it pulls Taylor Swift and The Weeknd's music


In [70]:
# Let's get an example
res = filter_by_metadata('title', ['Israel'], collection, limit = 2)
for x in res:
    print(x['title'])

Israel Supreme Court strikes down judicial reforms
Israel to fight South Africa's Gaza genocide claim in court


<a id='ex02'></a>

<a id='3-2'></a>
### 3.2 Semantic search

<a id='ex02'></a>
### Exercise 2

In this exercise, you will implement a semantic search retrieval, similar to the one you created in the previous assignment, but this time utilizing the Weaviate API.

<details>
<summary style="color: green;">Hint</summary>
<p>Remember that to perform semantic search, you should use the method <code>collection.query.near_text</code>.</p>
<p>The <code>top_k</code> parameter in the function dictates how many results to retrieve. In Weaviate, this is referred to as <code>limit</code>. Adjust this parameter as needed.</p>
</details>

In [71]:
# GRADED CELL 

def semantic_search_retrieve(query: str,
                             collection: "weaviate.collections.collection.sync.Collection" , 
                             top_k: int = 5) -> list:
    """
    Performs a semantic search on a collection and retrieves the top relevant chunks.

    This function executes a semantic search query on a specified collection to find text chunks 
    that are most relevant to the input 'query'. The search retrieves a limited number of top 
    matching objects, as specified by 'top_k'. The function returns the 'chunk' property of 
    each of the top matching objects.

    Args:
    query (str): The search query used to find relevant text chunks.
    collection (weaviate.collections.collection.sync.Collection): The collection in which the semantic search is performed.
    top_k (int, optional): The number of top relevant objects to retrieve. Defaults to 5.

    Returns:
    List[str]: A list of text chunks that are most relevant to the given query.
    """
    ### START CODE HERE ###

    # Retrieve using collection.query.near_text
    response = collection.query.near_text(
        query=query,
        limit=top_k
    )

    ### END CODE HERE ###
    
    response_objects = [x.properties for x in response.objects]
    
    return response_objects

In [80]:
# Let's have an example!
response = semantic_search_retrieve(query = 'Isreal and gaza', collection = collection, top_k = 2)

In [81]:
response[1]['title']

'Israel-Palestinian bitterness deepened by Hamas attack and war'

In [92]:
# Let's have an example!
result = semantic_search_retrieve(query = 'Tell me about the last Taylor Swift show', collection = collection, top_k = 2)

In [93]:
result[0]['title']

'Madonna sued by fans in New York over late concert start time'

In [94]:
'Taylor Swift' in result[0]['article_content']

False

<a id='ex03'></a>

<a id='3-3'></a>
### 3.3 BM25 Serach

<a id='ex03'></a>
### Exercise 3

In this exercise, you will implement a BM25 retrieval, similar to the one you created in the previous assignment, but now using the Weaviate API.
<details>
<summary style="color: green;">Hint</summary>
<p>To perform a BM25 search, use the method <code>collection.query.bm25</code>.</p>
<p>The <code>top_k</code> parameter in the function specifies how many results to retrieve. In Weaviate, this parameter is referred to as <code>limit</code>. Adjust this accordingly.</p>
</details>

In [83]:
# GRADED CELL 

def bm25_retrieve(query: str, 
                  collection: "weaviate.collections.collection.sync.Collection" , 
                  top_k: int = 5) -> list:
    """
    Performs a BM25 search on a collection and retrieves the top relevant chunks.

    This function executes a BM25-based search query on a specified collection to identify text 
    chunks that are most relevant to the provided 'query'. It retrieves a limited number of the 
    top matching objects, as specified by 'top_k', and returns the 'chunk' property of these objects.

    Args:
    query (str): The search query used to find relevant text chunks.
    collection (weaviate.collections.collection.sync.Collection): The collection in which the BM25 search is performed.
    top_k (int, optional): The number of top relevant objects to retrieve. Defaults to 5.

    Returns:
    List[str]: A list of text chunks that are most relevant to the given query.
    """
    
    ### START CODE HERE ###

    # Retrieve using collection.query.bm25
    response = collection.query.bm25(
        query=query,
        limit=top_k)

    ### END CODE HERE ### 
    
    response_objects = [x.properties for x in response.objects]
    return response_objects 

In [84]:
result = bm25_retrieve('Tell me about the last Taylor Swift show', collection, top_k = 2)

In [86]:
result[0]['title']

'Margot Robbie, Taylor Swift and more on Golden Globes red carpet'

<a id='ex04'></a>

<a id='3-4'></a>
### 3.4 Hybrid search

<a id='ex04'></a>
### Exercise 4

In this exercise, you will implement a Reciprocal Rank Fusion (RRF) retrieval system using the Weaviate API. To achieve this, you will need to use the `collection.query.hybrid` method.



<details>
<summary style="color: green;">Hint</summary>
<p>To perform a hybrid search, use the method <code>collection.query.hybrid</code>.</p>
<p>The <code>top_k</code> parameter in the function specifies how many results to retrieve. In Weaviate, this is referred to as <code>limit</code>. Make sure to also include the <code>alpha</code>.</p>
</details>

In [95]:
# GRADED CELL 

def hybrid_retrieve(query: str, 
                    collection: "weaviate.collections.collection.sync.Collection" , 
                    alpha: float = 0.5,
                    top_k: int = 5
                   ) -> list:
    """
    Performs a hybrid search on a collection and retrieves the top relevant chunks.

    This function executes a hybrid search that combines semantic vector search and traditional 
    keyword-based search on a specified collection to find text chunks most relevant to the 
    input 'query'. The relevance of results is influenced by 'alpha', which balances the weight 
    between vector and keyword matches. It retrieves a limited number of top matching objects, 
    as specified by 'top_k', and returns the 'chunk' property of these objects.

    Args:
    query (str): The search query used to find relevant text chunks.
    collection (weaviate.collections.collection.sync.Collection): The collection in which the hybrid search is performed.
    alpha (float, optional): A weighting factor that balances the contribution of semantic 
    and keyword matches. Defaults to 0.5.
    top_k (int, optional): The number of top relevant objects to retrieve. Defaults to 5.

    Returns:
    List[str]: A list of text chunks that are most relevant to the given query.
    """
    ### START CODE HERE ### 

    # Retrieve using collection.query.hybrid
    response = collection.query.hybrid(
        query=query,
        alpha=alpha,
        limit=top_k
    )

    ### END CODE HERE ###
    
    response_objects = [x.properties for x in response.objects]
    
    return response_objects 

In [96]:
result = hybrid_retrieve('Tell me about the last Taylor Swift show', collection, top_k = 2)

In [98]:
result[0]['title']

'Margot Robbie, Taylor Swift and more on Golden Globes red carpet'

### 5 - Reranking

<a id='ex05'></a>

<a id='ex05'></a>
### Exercise 5

In this section, you will create a new version of `semantic_search` that allows reranking of the results. This new function must support using a different query for reranking or reranking based on a specific document property (e.g., reranking using only the title).

Your task is to add the `rerank` parameter to the `collection.query.near_text` call.

<details>
<summary style="color: green;">Hint 1</summary>
<p>Remember that <code>collection.query.near_text</code> takes a query, a limit (i.e., <code>top_k</code>), and now also requires the <code>rerank</code> parameter.</p>
</details>

<details>
<summary style="color: green;">Hint 2</summary>
<p>The <code>Rerank</code> object is already loaded into memory. It takes two parameters: the query and the document property to use for ranking—<code>query</code> and <code>prop</code>, respectively.</p>
</details>

<details>
<summary style="color: green;">Hint 3</summary>
<p>Define the reranker as <code>reranker = Reranker(appropriate_parameters)</code>. Don’t forget: the query for the reranker should be <code>rerank_query</code>!</p>
</details>

In [99]:
# GRADED CELL 

def semantic_search_with_reranking(query: str, 
                                   rerank_property: str,
                                   collection: "weaviate.collections.collection.sync.Collection" , 
                                   rerank_query: str = None,
                                   top_k: int = 5
                                   ) -> list:
    """
    Performs a semantic search and reranks the results based on a specified property.

    Args:
        query (str): The search query to perform the initial search.
        rerank_property (str): The property used for reranking the search results.
        collection (weaviate.collections.collection.sync.Collection): The collection to search within.
        rerank_query (str, optional): The query to use specifically for reranking. If not provided, 
                                      the original query is used for reranking.
        top_k (int, optional): The maximum number of top results to return. Defaults to 5.

    Returns:
        list: A list of properties from the reranked search results, where each item corresponds to 
              an object in the collection.
    """
    ### START CODE HERE ### 

    # Set the rerank_query to be the same as the query if rerank_query is not passed (don't change this line)
    if rerank_query is None: 
        rerank_query = query 
        
    # Define the reranker with rerank_query and rerank_property
    reranker = Rerank(
        prop=rerank_property,
        query=rerank_query
    )

    # Retrieve using collection.query.near_text with the appropriate parameters (do not forget the rerank!)
    response = collection.query.near_text(
        query=query,
        limit=top_k,
        rerank=reranker
    )

    ### END CODE HERE ###
    
    response_objects = [x.properties for x in response.objects]
    
    return response_objects 

I do not have a **reranker**

In [41]:
client.close()

In [42]:
!docker-compose down -v

/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=30520) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


[+] Running 0/1
 ⠋ Container graded_lab-weaviate-1  S...                                   0.1s 
[+] Running 0/1
 ⠙ Container graded_lab-weaviate-1  S...                                   0.2s 
[+] Running 0/1
 ⠹ Container graded_lab-weaviate-1  S...                                   0.3s 
[+] Running 0/1
 ⠸ Container graded_lab-weaviate-1  S...                                   0.4s 
[+] Running 0/1
 ⠼ Container graded_lab-weaviate-1  S...                                   0.5s 
[+] Running 0/1
 ⠴ Container graded_lab-weaviate-1  S...                                   0.6s 
[+] Running 0/1
 ⠦ Container graded_lab-weaviate-1  S...                                   0.7s 
[+] Running 0/1
 ⠧ Container graded_lab-weaviate-1  S...                                   0.8s 
[+] Running 0/1
 ⠇ Container graded_lab-weaviate-1  S...                                   0.9s 
[+] Running 0/1
 ⠏ Container graded_lab-weaviate-1  S...                                   1.0s 
[+] Running 0/1
 ⠋ Container g